# Dust devil likelihood: overlay preview without matplotlib

Same side-by-side check as `dust_devil_overlay_preview.ipynb` (CTX imagery left, hillshade right, likely areas in #EF4444), drawn with **numpy + Pillow only**. matplotlib crashed this kernel (exit code 3228369023 = 0xC06D007F, a broken native library) so it is not imported here.

Run `dust_devil_likelihood_demo.ipynb` first. Reads the npz, `rover_sol<N>_table.csv`, the meta.json and two .png/.jpg previews. Writes `likelihood_overlay_sol<N>.png`.

In [8]:
from pathlib import Path
import json
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFont
Image.MAX_IMAGE_PIXELS = None

PROC = Path("../processed_data")
DEMO_SOL = 800
IMAGERY_PREVIEW   = Path("../gale_tiles/out_tiles/imagery_preview_z11.png")        # SET THIS: zoom-11 CTX imagery preview (.png/.jpg)
HILLSHADE_PREVIEW = Path("../gale_tiles/out_tiles/topo_preview_z11.png")   # SET THIS: zoom-11 hillshade preview (.png/.jpg)
PREVIEW_BOUNDS = None    # (west, east, south, north) in degrees if the previews cover MORE than the template; None = same as template
LIKELY_TOP_PCT = 10      # highlight the top N% of pixels by daily-mean score (relative score, not a probability)
HIGHLIGHT = "#EF4444"
ALPHA = 0.55
PANEL_W = 1100           # width of each panel in the side-by-side image, in pixels
SEPARATE_W = 2000        # width of each separate preview image, in pixels

In [9]:
meta = json.loads((PROC / "gale_dust_devil_meta.json").read_text())
z = np.load(PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.npz")
likelihood_mean = z["likelihood_mean"]; tau = float(z["tau"]); day_ltst = tuple(z["day_ltst"]); r, c = int(z["rover_row"]), int(z["rover_col"])
H, W = likelihood_mean.shape
assert [H, W] == meta["template_shape"], "npz and meta.json come from different grids"
a, _, x0, _, e, y0 = meta["template_transform"]
west, east, south, north = x0, x0 + a*W, y0 + e*H, y0
rover_lon, rover_lat = meta["rover_lon"], meta["rover_lat"]
assert abs((x0 + (c + 0.5)*a) - rover_lon) < a and abs((y0 + (r + 0.5)*e) - rover_lat) < abs(e), "rover pixel in the npz does not contain the rover"

# model vs the rover's observed ground-air gap (numbers instead of a plot)
tbl = pd.read_csv(PROC / f"rover_sol{DEMO_SOL}_table.csv")
ok = tbl.gap_obs_k.notna()
x, y = tbl.gap_model_k_rover[ok].to_numpy(float), tbl.gap_obs_k[ok].to_numpy(float)
print("model vs observed gap at the rover: corr %.2f | peak observed %.1f K at %.2f h | peak model %.1f K at %.2f h" % (
    np.mean((x-x.mean())*(y-y.mean()))/(x.std()*y.std()), y.max(), tbl.ltst_bin_center_hr[ok].iloc[y.argmax()], x.max(), tbl.ltst_bin_center_hr[ok].iloc[x.argmax()]))
print("score range (daily mean): %.4f .. %.4f" % (np.nanmin(likelihood_mean), np.nanmax(likelihood_mean)))

model vs observed gap at the rover: corr 0.98 | peak observed 28.0 K at 12.38 h | peak model 28.0 K at 12.12 h
score range (daily mean): 0.0000 .. 0.9157


In [10]:
PH = round(PANEL_W * H / W)
rgb = tuple(int(HIGHLIGHT[i:i+2], 16) for i in (1, 3, 5))

thr = np.nanpercentile(likelihood_mean, 100 - LIKELY_TOP_PCT)
likely = np.isfinite(likelihood_mean) & (likelihood_mean >= thr)
def make_mask(w):
    return np.asarray(Image.fromarray(likely.astype("uint8") * 255).resize((w, round(w * H / W)), Image.NEAREST), dtype="float32")[..., None] / 255
mask = make_mask(PANEL_W)
print(f"top {LIKELY_TOP_PCT}% threshold: score >= {thr:.3f} ({int(likely.sum()):,} pixels)")

def panel(path, name, w=PANEL_W):
    ph = round(w * H / W); m = mask if w == PANEL_W else make_mask(w)
    assert path.exists(), f"set the preview path in the config cell; not found: {path}"
    assert path.suffix.lower() in (".png", ".jpg", ".jpeg"), f"{path.name}: export the preview as .png or .jpg"
    img = Image.open(path).convert("RGB")
    if PREVIEW_BOUNDS is not None:                                   # crop the preview to the template extent
        bw, be, bs, bn = PREVIEW_BOUNDS
        assert bw <= west and be >= east and bs <= south and bn >= north, "PREVIEW_BOUNDS must contain the template"
        sx, sy = img.width / (be - bw), img.height / (bn - bs)
        img = img.crop((round((west-bw)*sx), round((bn-north)*sy), round((east-bw)*sx), round((bn-south)*sy)))
    print(f"{name}: {path.name}, {img.width}x{img.height} px; aspect {img.width/img.height:.3f} vs template {W/H:.3f}")
    base = np.asarray(img.resize((w, ph), Image.LANCZOS), dtype="float32")
    out = base * (1 - ALPHA * m) + np.array(rgb, "float32") * (ALPHA * m)
    im = Image.fromarray(out.round().astype("uint8"))
    d = ImageDraw.Draw(im)
    px, py = (rover_lon - west) / (east - west) * w, (north - rover_lat) / (north - south) * ph
    rr_ = max(10, w // 110)
    d.ellipse([px-rr_, py-rr_, px+rr_, py+rr_], outline=(0, 0, 0), width=rr_ // 2 + 1); d.ellipse([px-rr_, py-rr_, px+rr_, py+rr_], outline=(0, 255, 255), width=max(3, rr_ // 4))
    return im

font = ImageFont.load_default()
HEAD, FOOT, GAP = 54, 40, 12
canvas = Image.new("RGB", (2*PANEL_W + GAP, HEAD + PH + FOOT), "white")
for k, (path, name) in enumerate([(IMAGERY_PREVIEW, "CTX imagery"), (HILLSHADE_PREVIEW, "Hillshade")]):
    canvas.paste(panel(path, name), (k*(PANEL_W + GAP), HEAD))
d = ImageDraw.Draw(canvas)
d.text((8, 8), f"Dust devil likelihood heuristic, sol {DEMO_SOL}, mean {day_ltst[0]:g}-{day_ltst[1]:g} LTST, tau={tau} (placeholder). Relative score, not a probability.", fill="black", font=font)
d.text((8, 28), f"Left: CTX imagery   Right: hillshade   |   lon {west:.3f}..{east:.3f} E, lat {south:.3f}..{north:.3f}", fill="black", font=font)
d.rectangle([8, HEAD+PH+12, 28, HEAD+PH+28], fill=rgb)
d.text((36, HEAD+PH+14), f"top {LIKELY_TOP_PCT}% heuristic score (>= {thr:.2f})", fill="black", font=font)
d.ellipse([360, HEAD+PH+10, 380, HEAD+PH+30], outline=(0, 200, 200), width=3)
d.text((388, HEAD+PH+14), f"rover, sol {DEMO_SOL}", fill="black", font=font)
out_png = PROC / f"likelihood_overlay_sol{DEMO_SOL}.png"
canvas.save(out_png)
print("saved", out_png, canvas.size)

top 10% threshold: score >= 0.602 (212,083 pixels)
CTX imagery: imagery_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
Hillshade: topo_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800.png (2212, 1163)


In [11]:
# ---- Separate preview images: one per background, same highlight, larger and uncluttered ----
for path, name, tag in [(IMAGERY_PREVIEW, "CTX imagery", "imagery"), (HILLSHADE_PREVIEW, "Hillshade", "hillshade")]:
    im = panel(path, name, w=SEPARATE_W)
    head, foot = 56, 48
    sheet = Image.new("RGB", (im.width, head + im.height + foot), "white")
    sheet.paste(im, (0, head))
    d = ImageDraw.Draw(sheet)
    d.text((8, 8), f"{name}: dust devil likelihood heuristic, sol {DEMO_SOL}, mean {day_ltst[0]:g}-{day_ltst[1]:g} LTST, tau={tau} (placeholder). Relative score, not a probability.", fill="black", font=font)
    d.text((8, 30), f"lon {west:.3f}..{east:.3f} E, lat {south:.3f}..{north:.3f}", fill="black", font=font)
    y = head + im.height + 14
    d.rectangle([8, y, 28, y+16], fill=rgb); d.text((36, y+2), f"top {LIKELY_TOP_PCT}% heuristic score (>= {thr:.2f})", fill="black", font=font)
    d.ellipse([360, y-2, 380, y+18], outline=(0, 200, 200), width=3); d.text((388, y+2), f"rover, sol {DEMO_SOL}", fill="black", font=font)
    out = PROC / f"likelihood_overlay_sol{DEMO_SOL}_{tag}.png"
    sheet.save(out); print("saved", out, sheet.size)

CTX imagery: imagery_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800_imagery.png (2000, 2048)
Hillshade: topo_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800_hillshade.png (2000, 2048)
